# Lab 5: Radar time series for tanks, and tanker draught from AIS

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

This lab has two parts. Both are about getting volume information when optical imagery is not enough.

**Part A: Sentinel-1 radar.** Build a backscatter time series for each floating-roof tank in Google Earth Engine, then design a test that checks it against optical fill estimates.

**Part B: Tanker draught from AIS.** Turn the draught that tankers broadcast into a rough estimate of the cargo they load or discharge at a port.

**Read this first: what is and is not established**

- Radar cuts through cloud and works at night. That is its advantage over Labs 3 and 4.
- Sentinel-1 pixels are 10 m. A 60 to 100 m tank covers only a few of them. The published work that measures roof height from radar used higher-resolution radar. Companies that use Sentinel-1 for tank inventories report results for many tanks combined, and they do not publish their methods.
- **Nothing in Part A has been shown to work for single tanks at Benicia or any other site in this project.** Part A builds the data and the test. The test decides.
- No method that measures tanker draught directly from Sentinel-1 or from Planet imagery was found. Part B uses the draught that the crew enters into the AIS transmitter. That entry is manual, can be stale or wrong, and the ship can switch AIS off.
- The cargo formula in Part B is a rough linear approximation. Real errors are larger than the synthetic tests here suggest, and are not measured in this lab.

**What you need**

- Part A: an Earth Engine account and project (see Lab 4), plus the tank file for your site from Lab 2.
- Part B: nothing to start. The lab includes made-up test data. For real data, see the download notes in Part B.

In [ ]:
import os
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

REPO = Path("..").resolve()
AOI_DIR = REPO / "AOI"
TANK_DIR = REPO / "data" / "tanks"
AIS_DIR = REPO / "data" / "ais"
AIS_DIR.mkdir(parents=True, exist_ok=True)

SITE = "valero_benicia_usa"     # an AOI file name without "_aoi"

# Part A: Sentinel-1 backscatter for tanks

## Concepts you need first

**Radar imaging (SAR).** A radar satellite sends microwave pulses and records what bounces back. It does not need sunlight and the pulses pass through cloud.

**Backscatter.** The strength of the returned signal. Smooth surfaces (calm water) send the pulse away from the satellite and look dark. Rough surfaces and corner shapes send it back and look bright.

**Double bounce.** A pulse hits a flat surface, then a wall, then returns to the satellite. Where a tank's wall meets its floating roof, this can produce a bright line. The line's position and strength may change when the roof rises or falls. This is the idea behind published tank-level radar work. Whether it shows at 10 m pixels is untested here.

**dB vs. linear.** Backscatter is usually shown in decibels (dB), a log scale. You cannot average dB values directly. Convert to linear power, average, then convert back. This lab does that.

**Speckle.** SAR images look grainy because of interference. A single pixel is noisy. Averages over many pixels or many dates are more stable.

**Incidence angle and orbit.** The angle at which the radar hits the ground changes the return. Sentinel-1 flies the same ground track every 12 days per satellite, either heading north (ascending) or south (descending). To compare dates fairly, use only images from **one relative orbit and one direction**.

**Time series.** One number per tank per date. We compute it two ways: the inner disc of the tank (roof) and a ring around the wall.

**Ground truth and validation.** A backscatter number is not a volume until it is compared with something known. Here the comparison is the optical fill from Labs 3 and 4 on clear days. If backscatter tracks optical fill on held-out dates, it can fill cloudy gaps. If it does not, it should not be used.

**Confounders.** Rain can pool on a roof. Wind, nearby equipment and the roof's own structure change the return. A time series that "levels off" does not by itself mean the tank stopped changing.

## Step 1: Settings

In [ ]:
START_DATE = "2025-01-01"
END_DATE = "2026-10-01"
POLARIZATION = "VV"           # or "VH"

# Zones used for each tank, as a share of the tank radius.
INNER_FRACTION = 0.6          # inner disc: out to 60 percent of the radius
RING_INNER, RING_OUTER = 0.8, 1.2   # ring around the wall

MIN_DIAMETER_M = 40           # ignore tanks too small for 10 m pixels
EVENT_DATE = None             # e.g. "2026-04-25". Set only after you confirm the date.
CHUNK_DAYS = 60               # fetch results in windows to stay under Earth Engine limits

## Step 2: Connect and load the tanks

This uses the same connection steps as Lab 4. You need the tank file from Lab 2 (`data/tanks/tanks_<site>.geojson`).

In [ ]:
import ee
import geopandas as gpd

EE_PROJECT = os.environ.get("EE_PROJECT") or input("Earth Engine project id: ")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

tank_file = TANK_DIR / f"tanks_{SITE}.geojson"
if not tank_file.exists():
    raise FileNotFoundError(f"Run Lab 2 first. Missing {tank_file.name}")

tanks = gpd.read_file(tank_file)
tanks = tanks[tanks["diameter_m"] >= MIN_DIAMETER_M].reset_index(drop=True)
print(len(tanks), "tanks at least", MIN_DIAMETER_M, "m wide")
tanks[["osm_id", "diameter_m", "content"]].head()

## Step 3: Define the zones for each tank

For each tank we build two shapes in Earth Engine: an **inner disc** (the roof) and a **ring** just around the wall. Earth Engine's `buffer` makes a circle of a given radius in meters around a point.

In [ ]:
def make_zone_collections(tank_table):
    """Return two ee.FeatureCollections: inner discs and rings, one feature per tank."""
    inner, ring = [], []
    for _, t in tank_table.iterrows():
        center = ee.Geometry.Point([float(t["center_lon"]), float(t["center_lat"])])
        R = float(t["diameter_m"]) / 2
        props = {"osm_id": int(t["osm_id"])}
        inner.append(ee.Feature(center.buffer(INNER_FRACTION * R), props))
        ring_geom = center.buffer(RING_OUTER * R).difference(center.buffer(RING_INNER * R))
        ring.append(ee.Feature(ring_geom, props))
    return ee.FeatureCollection(inner), ee.FeatureCollection(ring)


inner_fc, ring_fc = make_zone_collections(tanks)

# A rectangle that covers all tanks, used to filter the image collection.
minx, miny, maxx, maxy = tanks.total_bounds
aoi_box = ee.Geometry.Rectangle([float(minx), float(miny), float(maxx), float(maxy)])

## Step 4: Choose one orbit

Count the images by relative orbit number and direction, then keep the group with the most images. Comparing dates within one group keeps the viewing angle fixed.

In [ ]:
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(aoi_box)
      .filterDate(START_DATE, END_DATE)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", POLARIZATION)))

# Pull the orbit labels of every image (a small download).
orbit_props = ["system:time_start", "relativeOrbitNumber_start", "orbitProperties_pass"]
rows = ee.FeatureCollection(s1.map(lambda im: ee.Feature(None, im.toDictionary(orbit_props)))
                            ).getInfo()["features"]
orbit_table = pd.DataFrame([r["properties"] for r in rows])
orbit_table["time"] = pd.to_datetime(orbit_table["system:time_start"], unit="ms", utc=True)

counts = (orbit_table.groupby(["relativeOrbitNumber_start", "orbitProperties_pass"])
          .size().sort_values(ascending=False))
print("Images per (relative orbit, direction):")
print(counts.head(6).to_string())

REL_ORBIT, PASS_DIR = counts.index[0]
print("\nUsing relative orbit", REL_ORBIT, PASS_DIR)

s1_one = (s1.filter(ee.Filter.eq("relativeOrbitNumber_start", int(REL_ORBIT)))
            .filter(ee.Filter.eq("orbitProperties_pass", PASS_DIR)))

## Step 5: Extract the time series

Earth Engine returns Sentinel-1 backscatter in dB. For each image we:

1. convert dB to linear power,
2. average the linear values inside each zone (`reduceRegions`),
3. keep the incidence angle too.

Results come back in date windows, so no single request is too large.

In [ ]:
def extract_zone(collection, zone_fc, zone_name, start, end):
    """Mean linear backscatter and incidence angle per tank for each image in a date window."""
    def per_image(img):
        linear = ee.Image(10).pow(img.select(POLARIZATION).divide(10)).rename("sigma_lin")
        stack = linear.addBands(img.select("angle"))
        stats = stack.reduceRegions(collection=zone_fc, reducer=ee.Reducer.mean(), scale=10)
        return stats.map(lambda f: f.set({"t": img.get("system:time_start"), "zone": zone_name}))
    sub = collection.filterDate(start, end)
    return ee.FeatureCollection(sub.map(per_image)).flatten().getInfo()["features"]


def extract_series(collection, zone_fc, zone_name):
    windows = pd.date_range(START_DATE, END_DATE, freq=f"{CHUNK_DAYS}D")
    windows = list(windows) + [pd.Timestamp(END_DATE)]
    frames = []
    for a, b in zip(windows[:-1], windows[1:]):
        feats = extract_zone(collection, zone_fc, zone_name, a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d"))
        if feats:
            frames.append(pd.DataFrame([f["properties"] for f in feats]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


inner_df = extract_series(s1_one, inner_fc, "inner")
ring_df = extract_series(s1_one, ring_fc, "ring")
raw_series = pd.concat([inner_df, ring_df], ignore_index=True)
raw_series["time"] = pd.to_datetime(raw_series["t"], unit="ms", utc=True)
print(len(raw_series), "tank-date-zone rows")

In [ ]:
# One row per tank and date: inner and ring in dB, plus their difference.
wide = raw_series.pivot_table(index=["osm_id", "time"], columns="zone",
                              values=["sigma_lin", "angle"], aggfunc="mean")
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
wide["inner_db"] = 10 * np.log10(wide["sigma_lin_inner"])     # linear -> dB, after averaging
wide["ring_db"] = 10 * np.log10(wide["sigma_lin_ring"])
wide["inner_minus_ring_db"] = wide["inner_db"] - wide["ring_db"]

series_file = TANK_DIR / f"s1_series_{SITE}.csv"
wide.to_csv(series_file, index=False)
print("Saved", series_file.name)
wide.head()

## Step 6: Look at it

Each line is one tank. Two things to check before you interpret any trend:

- How large is the scatter between neighboring dates? That is the noise level.
- Do all tanks move together? If they do, the cause may be weather or the satellite, not the tanks.

In [ ]:
def plot_series(column, title, n_tanks=8):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for oid in wide["osm_id"].unique()[:n_tanks]:
        d = wide[wide["osm_id"] == oid].sort_values("time")
        ax.plot(d["time"], d[column], marker="o", markersize=3, linewidth=0.8, label=str(oid))
    if EVENT_DATE:
        ax.axvline(pd.Timestamp(EVENT_DATE, tz="UTC"), color="red", linestyle="--", label="event")
    ax.set_ylabel("dB")
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=4)
    plt.tight_layout()
    plt.show()


plot_series("inner_db", f"{POLARIZATION} backscatter, tank interior (orbit {REL_ORBIT} {PASS_DIR})")
plot_series("inner_minus_ring_db", "Interior minus ring (reduces some scene-wide effects)")

In [ ]:
# Summary per tank: how many dates, how big is the range, and how noisy are neighbors?
def summarize(d):
    d = d.sort_values("time")
    v = d["inner_db"].dropna().values
    return pd.Series({
        "dates": len(v),
        "range_db": v.max() - v.min() if len(v) else np.nan,
        "median_step_db": np.median(np.abs(np.diff(v))) if len(v) > 1 else np.nan,
    })

tank_summary = wide.groupby("osm_id").apply(summarize)
tank_summary["range_over_step"] = tank_summary["range_db"] / tank_summary["median_step_db"]
tank_summary.sort_values("range_over_step", ascending=False).round(2).head(15)

**How to read the table.** `range_db` is the spread of the whole series. `median_step_db` is the typical change between neighboring dates, which is a rough noise level. A tank whose range is only a few times its step has a signal that is hard to tell from noise.

## Step 7: Validate against optical fill

This is the step that decides whether the radar series means anything. The plan:

1. Collect optical fill estimates from Labs 3 and 4. Each metrics file lists `osm_id`, a fill value and a scene id that begins with the date.
2. Pair each optical estimate with the nearest Sentinel-1 date for the same tank (within a few days).
3. Fit a line from backscatter to optical fill on part of the pairs. Test it on the rest.
4. Report how large the errors are, and whether the rank order of fills is preserved.

First the code. We test it on **made-up data** so you can see what a good and a bad relationship look like.

In [ ]:
def scene_date(scene_id):
    """Read the date from a scene id that starts with yyyymmdd."""
    return pd.Timestamp(str(scene_id)[:8], tz="UTC")


def pair_with_optical(sar, optical, max_days=2):
    """For each optical row, find the closest SAR date for the same tank.

    sar: columns osm_id, time, inner_db
    optical: columns osm_id, time, fill
    """
    pairs = []
    for _, o in optical.iterrows():
        s = sar[sar["osm_id"] == o["osm_id"]]
        if s.empty:
            continue
        gap = (s["time"] - o["time"]).abs()
        j = gap.idxmin()
        if gap.loc[j] <= pd.Timedelta(days=max_days):
            pairs.append({"osm_id": o["osm_id"], "optical_time": o["time"], "fill": o["fill"],
                          "sar_time": s.loc[j, "time"], "inner_db": s.loc[j, "inner_db"],
                          "gap_days": gap.loc[j] / pd.Timedelta(days=1)})
    return pd.DataFrame(pairs)


def validate(pairs, by_tank=True):
    """Leave-one-out test of a straight-line fit from backscatter to optical fill.

    by_tank=True fits one line per tank (each tank has its own geometry).
    Returns a summary dict.
    """
    preds, truth = [], []
    groups = pairs.groupby("osm_id") if by_tank else [("all", pairs)]
    for _, g in groups:
        g = g.dropna(subset=["fill", "inner_db"])
        if len(g) < 5:
            continue
        x, y = g["inner_db"].values, g["fill"].values
        for i in range(len(g)):
            keep = np.arange(len(g)) != i                       # leave one pair out
            slope, intercept = np.polyfit(x[keep], y[keep], 1)  # straight-line fit
            preds.append(slope * x[i] + intercept)
            truth.append(y[i])
    if not preds:
        return {"n": 0}
    preds, truth = np.array(preds), np.array(truth)
    rho = spearmanr(preds, truth).correlation
    return {"n": len(preds),
            "rmse_fill": float(np.sqrt(np.mean((preds - truth) ** 2))),
            "baseline_rmse": float(np.std(truth)),      # error of always guessing the average
            "spearman": float(rho)}

In [ ]:
# MADE-UP DATA. This tests the code above. It says nothing about real radar physics.
def fake_case(radar_noise_db, n_tanks=6, n_optical=12, seed=0):
    rng = np.random.default_rng(seed)
    days = pd.date_range("2025-01-01", "2026-09-30", freq="6D", tz="UTC")
    sar_rows, opt_rows = [], []
    for k in range(n_tanks):
        oid = 1000 + k
        fill = np.clip(0.5 + 0.35 * np.sin(np.arange(len(days)) / 9 + k), 0.05, 0.95)  # hidden truth
        db = -8 + 4.0 * fill + rng.normal(0, radar_noise_db, len(days))                # assumed link
        sar_rows += [{"osm_id": oid, "time": t, "inner_db": v} for t, v in zip(days, db)]
        pick = rng.choice(len(days), n_optical, replace=False)                         # clear days
        opt_rows += [{"osm_id": oid, "time": days[i] + pd.Timedelta(days=int(rng.integers(-1, 2))),
                      "fill": float(np.clip(fill[i] + rng.normal(0, 0.05), 0, 1))} for i in pick]
    return pd.DataFrame(sar_rows), pd.DataFrame(opt_rows)


results = {}
for noise in (0.2, 0.5, 2.0, 5.0):
    fs, fo = fake_case(noise)
    results[noise] = validate(pair_with_optical(fs, fo))
pd.DataFrame(results).T.round(3).rename_axis("radar noise (dB)")

Read the four rows. With low noise, the error is well below the **baseline error**, which is what you would get by always guessing the average fill, and the rank order of fills is kept (Spearman close to 1). At 2 dB of noise and above, the error is no better than the baseline and the rank order is lost. When RMSE is no better than the baseline, the radar series adds no information.

Real data will not look like this made-up link. Use it as a standard for what "good enough" means before you look at your own results.

**Now run it on real data.** This needs the metrics files from Lab 3 or Lab 4 for the same site. If none exist, the cell says so.

In [ ]:
metric_files = sorted(TANK_DIR.glob(f"metrics_*{SITE}*.csv"))
frames = []
for f in metric_files:
    m = pd.read_csv(f)
    if "osm_id" not in m.columns or m["osm_id"].isna().all():
        continue
    fill_col = "fill_fit" if "fill_fit" in m.columns else "fill"      # prefer the fit method (Lab 4)
    frames.append(pd.DataFrame({"osm_id": m["osm_id"].astype("Int64"),
                                "time": m["scene"].map(scene_date), "fill": m[fill_col]}))
optical = pd.concat(frames, ignore_index=True).dropna() if frames else pd.DataFrame()
print(len(metric_files), "metrics files;", len(optical), "usable optical fill estimates")

if len(optical):
    optical["osm_id"] = optical["osm_id"].astype(int)
    pairs = pair_with_optical(wide, optical)
    print(len(pairs), "optical/radar pairs")
    print(validate(pairs))
else:
    print("No optical estimates with tank ids yet. Run Labs 3 or 4 on several dates first.")

**Interpreting a real result.** You need many pairs per tank. With fewer than about 5 pairs, the code skips the tank. A handful of clear days per year may not be enough. If so, the finding is that this approach cannot be validated with the data available, which is itself a useful result.

## Step 8: Where Sentinel-1 helps without being exact

Even if single-tank volume fails validation, Sentinel-1 can still:

- tell you whether a tank is floating-roof or fixed-roof (the signature differs),
- show whether many tanks at a site change together (a regional index),
- keep a record through cloudy months,
- detect ships at berth (used in Part B).

Companies that sell tank inventories use these properties and calibrate against known inventories. You can do that only where public inventory data exists.

# Part B: Tanker draught from AIS

## Concepts you need first

**AIS (Automatic Identification System).** Ships broadcast their identity, position, speed and some details. Receivers on shore and in orbit collect this. Large ships must carry it.

**MMSI and IMO numbers.** Two ship identifiers. The MMSI identifies the radio. The IMO number identifies the hull and stays with it for life.

**Draught (draft).** The depth from waterline to keel. A loaded tanker sits deeper than an empty one. A very large crude carrier (VLCC) may sit around 20 m or more deep when full and around 8 to 10 m when in ballast.

**The draught field.** The AIS standard stores draught in tenths of a meter. The value 0 means "not available". The value 25.5 means 25.5 m or more. The crew enters it by hand. It is often not updated.

**Ballast.** Empty tankers take on seawater to stay stable. So an empty tanker still has a draught well above zero.

**Deadweight tonnage (DWT).** The weight a ship can carry, including cargo, fuel and stores. It is a design value for the ship at its maximum draught.

**Rough cargo estimate.** If draught changes by ΔT at a port, and the ship's full-load and ballast draughts are `T_max` and `T_ballast`:

`cargo change ≈ DWT × ΔT ÷ (T_max − T_ballast)`

This treats the hull as a straight-sided box. Real hulls are not. It ignores fuel, ballast water and stores. Treat results as rough.

**Vessel class.** Ships of different sizes have different DWT and draughts. We guess the class from ship length. The numbers in the table below are **illustrative round figures**. Replace them with registry data for any real analysis.

**Port call.** A period when a ship is stopped at or near a terminal. We look at draught before it arrives and after it leaves.

In [ ]:
# ILLUSTRATIVE round numbers. Replace with data for the real ships.
vessel_classes = pd.DataFrame([
    # name,     min length (m), DWT (tonnes), full draught (m), ballast draught (m)
    ("VLCC",       300, 300_000, 21.0, 9.0),
    ("Suezmax",    250, 160_000, 16.5, 8.0),
    ("Aframax",    220, 110_000, 14.5, 7.0),
    ("Panamax",    180,  70_000, 13.0, 6.5),
    ("MR",           0,  50_000, 12.0, 6.0),
], columns=["class", "min_length_m", "dwt_t", "t_full_m", "t_ballast_m"])

BARRELS_PER_TONNE = 7.3     # typical crude oil; refined products differ
vessel_classes

## Step 1: Load AIS data in one standard layout

Two free sources, with different column names:

- **United States:** MarineCadastre / NOAA. Columns include `MMSI`, `BaseDateTime`, `LAT`, `LON`, `SOG`, `VesselType`, `Draft`, `IMO`, `Length`. Tankers have vessel type 80 to 89.
- **Denmark:** Danish Maritime Authority (aisdata.ais.dk). Columns include `# Timestamp`, `MMSI`, `Latitude`, `Longitude`, `SOG`, `Ship type`, `IMO`, `Length`, `Draught`.

`standardize_ais` renames either one to the same layout. AIS files are large, so `read_ais_csv` reads in chunks and keeps only points inside a box around your port.

In [ ]:
def standardize_ais(raw, source):
    """Convert a raw AIS table ('us' or 'dk') to a standard layout."""
    if source == "us":
        return pd.DataFrame({
            "mmsi": raw["MMSI"],
            "time": pd.to_datetime(raw["BaseDateTime"], utc=True, errors="coerce"),
            "lat": raw["LAT"], "lon": raw["LON"], "sog": raw["SOG"],
            "is_tanker": pd.to_numeric(raw["VesselType"], errors="coerce").between(80, 89),
            "draught_m": pd.to_numeric(raw["Draft"], errors="coerce"),
            "imo": raw["IMO"],
            "length_m": pd.to_numeric(raw["Length"], errors="coerce")})
    if source == "dk":
        return pd.DataFrame({
            "mmsi": raw["MMSI"],
            "time": pd.to_datetime(raw["# Timestamp"], dayfirst=True, utc=True, errors="coerce"),
            "lat": raw["Latitude"], "lon": raw["Longitude"], "sog": raw["SOG"],
            "is_tanker": raw["Ship type"].astype(str).str.contains("tanker", case=False),
            "draught_m": pd.to_numeric(raw["Draught"], errors="coerce"),
            "imo": raw["IMO"],
            "length_m": pd.to_numeric(raw["Length"], errors="coerce")})
    raise ValueError("source must be 'us' or 'dk'")


def read_ais_csv(path, source, bbox, chunk_rows=500_000):
    """Read a large AIS file in chunks and keep points inside bbox = (west, south, east, north)."""
    west, south, east, north = bbox
    kept = []
    for chunk in pd.read_csv(path, chunksize=chunk_rows, low_memory=False):
        std = standardize_ais(chunk, source)
        inside = std["lon"].between(west, east) & std["lat"].between(south, north)
        kept.append(std[inside & std["is_tanker"]])
    return pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()


def valid_draught(d):
    """Keep draught values that are real readings (not 0 = unavailable, not the 25.5 cap)."""
    return d.where((d > 0) & (d < 25.5))

## Step 2: Find port calls and read the draught change

For each tanker the code:

1. keeps points within a set distance of the port,
2. finds periods when the ship is nearly stopped (speed under 0.5 knots) for at least 6 hours,
3. takes the draught readings in the 12 hours before the stop (arrival) and in the 12 hours after it (departure),
4. computes the change and converts it to a rough cargo change.

A negative change means the ship got shallower, so it discharged cargo. A positive change means it loaded.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between points on the globe."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


def find_port_calls(ais, center_lat, center_lon, radius_km, stop_kn=0.5,
                    min_stop_h=6, merge_gap_h=6, window_h=12):
    """List port calls with draught on arrival and on departure."""
    dist = haversine_km(ais["lat"], ais["lon"], center_lat, center_lon)
    near = ais[(dist <= radius_km) & ais["is_tanker"]].copy()
    near["draught_ok"] = valid_draught(near["draught_m"])
    calls = []
    for mmsi, g in near.sort_values("time").groupby("mmsi"):
        stopped = g[g["sog"] < stop_kn]
        if stopped.empty:
            continue
        # A new stop begins whenever there is a long gap between stopped points.
        stop_id = (stopped["time"].diff() > pd.Timedelta(hours=merge_gap_h)).cumsum()
        for _, s in stopped.groupby(stop_id):
            start, end = s["time"].min(), s["time"].max()
            if end - start < pd.Timedelta(hours=min_stop_h):
                continue
            before = g[(g["time"] <= start) & (g["time"] >= start - pd.Timedelta(hours=window_h))]
            after = g[(g["time"] >= end) & (g["time"] <= end + pd.Timedelta(hours=window_h))]
            calls.append({
                "mmsi": mmsi, "imo": g["imo"].iloc[0], "length_m": g["length_m"].median(),
                "arrive": start, "depart": end, "hours": (end - start) / pd.Timedelta(hours=1),
                "draught_arrival_m": before["draught_ok"].median(),
                "draught_departure_m": after["draught_ok"].median()})
    return pd.DataFrame(calls)


def assign_class(length_m):
    """Pick the largest class whose minimum length the ship meets."""
    if pd.isna(length_m):
        return None
    ok = vessel_classes[vessel_classes["min_length_m"] <= length_m]
    return ok.sort_values("min_length_m").iloc[-1]["class"]


def estimate_cargo_change(calls):
    """Add class, draught change, rough cargo change and quality flags to the call table."""
    out = calls.copy()
    out["class"] = out["length_m"].map(assign_class)
    out = out.merge(vessel_classes, on="class", how="left")
    out["delta_draught_m"] = out["draught_departure_m"] - out["draught_arrival_m"]
    fraction = out["delta_draught_m"] / (out["t_full_m"] - out["t_ballast_m"])
    out["cargo_change_t"] = (out["dwt_t"] * fraction).clip(-out["dwt_t"], out["dwt_t"])
    out["cargo_change_bbl"] = out["cargo_change_t"] * BARRELS_PER_TONNE

    flags = []
    for _, r in out.iterrows():
        f = []
        if pd.isna(r["delta_draught_m"]):
            f.append("missing_draught")
        else:
            if abs(r["delta_draught_m"]) < 0.3:
                f.append("no_change_detected_or_not_updated")
            for t in (r["draught_arrival_m"], r["draught_departure_m"]):
                if pd.notna(t) and not (r["t_ballast_m"] - 2 <= t <= r["t_full_m"] + 1):
                    f.append("draught_outside_class_range")
                    break
        flags.append(";".join(f))
    out["flags"] = flags
    return out

## Step 3: Test on made-up data

No real AIS file is needed yet. The generator below creates tankers that arrive, stop for a day or two and leave, in the US file layout. Each call has a known true cargo change. Some departure draughts are **not updated** (the crew forgot) and some readings are 0 (unavailable). That lets you see the failure modes.

This tests the code. It does **not** test how well a real ship's draught maps to its cargo, because the generator uses the same straight-line formula the estimator uses.

In [ ]:
def make_fake_ais(n_calls=24, center=(38.05, -122.15), stale_prob=0.2, missing_prob=0.1,
                  draught_noise_m=0.2, seed=3):
    """Return raw AIS rows (US layout) and a table of true cargo changes."""
    rng = np.random.default_rng(seed)
    rows, truth = [], []
    for k in range(n_calls):
        cls = vessel_classes.sample(1, random_state=int(rng.integers(1e6))).iloc[0]
        length = cls["min_length_m"] + 15 if cls["min_length_m"] else 180
        f_arr, f_dep = rng.uniform(0.05, 0.95, 2)
        t_arr = cls["t_ballast_m"] + f_arr * (cls["t_full_m"] - cls["t_ballast_m"])
        t_dep = cls["t_ballast_m"] + f_dep * (cls["t_full_m"] - cls["t_ballast_m"])
        stale = rng.random() < stale_prob                         # crew did not update on departure
        start = pd.Timestamp("2026-03-01", tz="UTC") + pd.Timedelta(days=3 * k)
        stop_h = float(rng.uniform(10, 40))
        end = start + pd.Timedelta(hours=stop_h)
        times = pd.date_range(start - pd.Timedelta(hours=10), end + pd.Timedelta(hours=10), freq="30min")
        for t in times:
            moving = (t < start) or (t > end)
            switched = t > end - pd.Timedelta(hours=2)            # draught is updated just before sailing
            draught = t_dep if (switched and not stale) else t_arr
            draught = round(draught + rng.normal(0, draught_noise_m), 1)
            if rng.random() < missing_prob:
                draught = 0.0                                     # "not available"
            rows.append({"MMSI": 200000000 + k, "BaseDateTime": t.strftime("%Y-%m-%dT%H:%M:%S"),
                         "LAT": center[0] + rng.normal(0, 0.01), "LON": center[1] + rng.normal(0, 0.01),
                         "SOG": 8.0 if moving else 0.1, "VesselType": 80 + int(rng.integers(0, 4)),
                         "Draft": draught, "IMO": 9000000 + k, "Length": float(length)})
        truth.append({"mmsi": 200000000 + k, "class": cls["class"],
                      "true_cargo_change_t": cls["dwt_t"] * (f_dep - f_arr), "stale": stale})
    return pd.DataFrame(rows), pd.DataFrame(truth)


raw_fake, truth_fake = make_fake_ais()
ais_fake = standardize_ais(raw_fake, "us")
calls_fake = find_port_calls(ais_fake, 38.05, -122.15, radius_km=6)
est_fake = estimate_cargo_change(calls_fake).merge(truth_fake, on="mmsi", suffixes=("", "_true"))
est_fake["error_t"] = est_fake["cargo_change_t"] - est_fake["true_cargo_change_t"]
print(len(calls_fake), "port calls found from", raw_fake["MMSI"].nunique(), "ships")
est_fake[["mmsi", "class", "draught_arrival_m", "draught_departure_m", "cargo_change_t",
          "true_cargo_change_t", "error_t", "stale", "flags"]].round(1).head(12)

In [ ]:
ok = est_fake[~est_fake["flags"].str.contains("missing")]
fresh, stale = ok[~ok["stale"]], ok[ok["stale"]]
print(f"Calls with a usable estimate: {len(ok)} of {len(est_fake)}")
print(f"Calls where the crew updated draught:     median |error| = {fresh['error_t'].abs().median():>9,.0f} t  (n={len(fresh)})")
print(f"Calls where draught was NOT updated:      median |error| = {stale['error_t'].abs().median():>9,.0f} t  (n={len(stale)})")
print(f"Net cargo change, estimated: {ok['cargo_change_t'].sum():>12,.0f} t")
print(f"Net cargo change, true:      {ok['true_cargo_change_t'].sum():>12,.0f} t")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(ok["true_cargo_change_t"] / 1e3, ok["cargo_change_t"] / 1e3,
           c=ok["stale"].map({True: "tab:red", False: "tab:blue"}))
lim = ok[["true_cargo_change_t", "cargo_change_t"]].abs().max().max() / 1e3 * 1.05
ax.plot([-lim, lim], [-lim, lim], color="gray", linestyle=":")
ax.set_xlabel("True cargo change (thousand tonnes)")
ax.set_ylabel("Estimated (thousand tonnes)")
ax.set_title("Made-up data. Red = draught not updated by crew")
plt.show()

**What the test shows.** When the crew updates the draught, the estimate follows the truth within a few thousand tonnes. When the crew does not, the estimate reads near zero even if a full cargo moved. In real data you cannot see which calls are stale. A `no_change_detected_or_not_updated` flag is a warning, not a finding.

**What the test cannot show.** The generator uses the same straight-line formula as the estimator, so the real-world hull-shape error, fuel and ballast-water error are absent. Real errors are larger.

## Step 4: Run on real AIS data

1. **United States.** Use the MarineCadastre AccessAIS tool or the daily files. Save CSV files in `data/ais/`. Set `AIS_SOURCE = "us"`.
2. **Denmark.** Download daily files from the Danish Maritime Authority. Set `AIS_SOURCE = "dk"`.

Files are large (hundreds of MB per day for the whole US). Download only the days and area you need. Before trusting any column, open a file and check the names match `standardize_ais`.

**Port location.** The default center is the AOI center, which is the refinery, not the wharf. Ships stop at the wharf, which may be several kilometers away. Look at where tankers stop in your data and set `PORT_CENTER` and `PORT_RADIUS_KM` to match.

In [ ]:
AIS_SOURCE = "us"          # "us" or "dk"

aoi_feature = json.loads((AOI_DIR / f"{SITE}_aoi.geojson").read_text())["features"][0]["properties"]
PORT_CENTER = (aoi_feature["center_lat"], aoi_feature["center_lon"])    # CHANGE to the wharf
PORT_RADIUS_KM = 8

lat0, lon0 = PORT_CENTER
dlat = PORT_RADIUS_KM / 111.0
dlon = PORT_RADIUS_KM / (111.0 * math.cos(math.radians(lat0)))
bbox = (lon0 - dlon, lat0 - dlat, lon0 + dlon, lat0 + dlat)

ais_files = sorted(AIS_DIR.glob("*.csv"))
print(len(ais_files), "AIS files in data/ais/")

if ais_files:
    ais_real = pd.concat([read_ais_csv(f, AIS_SOURCE, bbox) for f in ais_files], ignore_index=True)
    print(len(ais_real), "tanker position reports near the port,", ais_real["mmsi"].nunique(), "ships")
    calls_real = find_port_calls(ais_real, lat0, lon0, PORT_RADIUS_KM)
    result = estimate_cargo_change(calls_real)
    out_file = TANK_DIR.parent / "ais" / f"port_calls_{SITE}.csv"
    result.to_csv(out_file, index=False)
    print(len(result), "port calls. Saved", out_file.name)
    print(result["flags"].replace("", "ok").value_counts().to_string())
else:
    print("No AIS files yet. Download some (see above), then rerun this cell.")

In [ ]:
if ais_files and len(result):
    usable = result[~result["flags"].str.contains("missing")].copy()
    usable["month"] = usable["depart"].dt.to_period("M")
    monthly = usable.groupby("month").agg(calls=("mmsi", "size"),
                                          net_change_million_bbl=("cargo_change_bbl", lambda s: s.sum() / 1e6))
    print(monthly.round(2).to_string())

## Check your work

- In Part A, what is the validation RMSE compared with the baseline? If they are close, say so plainly.
- In Part B, what share of calls have a usable draught reading at both ends? What share are flagged as not updated?
- Take one large call and check by hand whether the draught values make sense for that ship's class.

## Exercises

1. In Step 7 (Part A), try noise levels of 1 and 3 dB. At what level does the radar stop beating the baseline?
2. Add a second tank group, a fixed-roof tank, to the Part A fake data. A fixed roof does not move with the oil. What should its validation look like?
3. In Part B, set `stale_prob` to 0.5 in `make_fake_ais`. How does the net cargo estimate change? What does this say about relying on draught for flow totals?
4. Change the vessel class table: raise the VLCC full draught to 22 m. How much does a typical VLCC estimate shift? Which input matters most?
5. A ship turns AIS off before arriving and back on after leaving. What can an imagery-based approach still tell you? (Think of Lab 4 and Sentinel-1 ship detection.)
6. Write a short memo (200 words) for a policymaker. How much weight should a draught-based cargo estimate carry against official export figures, and what independent checks would raise confidence?